<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


---

<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a> 所著《<a href="http://mng.bz/orYv">从零构建大语言模型</a>》一书的配套代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Memory-efficient Model Weight Loading


---

# 内存高效模型权重加载


- This notebook provides tips for loading larger pretrained or finetuned models when GPU (or CPU) memory is limited
- Specifically, it focuses on cases where you saved the model using `torch.save(model.state_dict(), "model.pth")` (for example, in chapters 5-7) and want to load it in a new session later for continued pretraining or additional finetuning
- While the example uses an LLM, the methods explained in this notebook are general and apply to loading any PyTorch model, not just LLMs


---

- 本笔记本提供在 GPU（或 CPU）内存有限时加载较大预训练或微调模型的技巧
- 具体而言，它专注于以下场景：您曾使用 `torch.save(model.state_dict(), "model.pth")` 保存模型（例如在第5-7章中），并希望在新会话中加载该模型以进行持续预训练（continued pretraining）或额外微调
- 虽然示例使用了大语言模型（LLM），但本笔记本中解释的方法具有通用性，适用于加载任何 PyTorch 模型，而不仅限于大语言模型


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/memory-efficient-loading/memory-efficient-loading.webp" width="800px">


---

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/memory-efficient-loading/memory-efficient-loading.webp" width="800px">


In [1]:
from importlib.metadata import version

pkgs = [
    "torch",
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

torch version: 2.9.1+cu130


&nbsp;
## 1. Benchmark utilities


---

&nbsp;
## 1. 基准测试工具


- First, let's define some utility code to track VRAM (GPU memory)
- Later, we will also introduce a tool to track the main system RAM (CPU memory)
- The purpose of these functions will become clear when we apply them later


---

- 首先，我们将定义一些用于追踪显存（GPU内存）的实用代码
- 后续我们还将引入一个追踪主系统内存（CPU内存）的工具
- 这些函数的具体用途将在后续应用时变得清晰


In [2]:
import gc
import time
import torch


def start_memory_tracking():
    """Initialize GPU memory tracking."""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    else:
        print("This notebook is intended for CUDA GPUs but CUDA is not available.")

def print_memory_usage():
    max_gpu_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)  # Convert bytes to GB
    print(f"Maximum GPU memory allocated: {max_gpu_memory:.1f} GB")

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(3)  # some buffer time to allow memory to clear
    torch.cuda.reset_peak_memory_stats()
    max_memory_allocated = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
    print(f"Maximum GPU memory allocated: {max_memory_allocated:.1f} GB")

&nbsp;
## 2. Model setup


---

&nbsp;
## 2. 模型设置


- This code section sets up the model itself
- Here, we use the "large" GPT-2 model to make things more interesting (you may use the "gpt2-small (124M)" to lower the memory requirements and execution time of this notebook)


---

- 此代码部分用于设置模型本身
- 在此我们使用"large"版本的GPT-2模型以增加趣味性（您可选用"gpt2-small (124M)"版本来降低本笔记本的内存需求与执行时间）


In [3]:
from previous_chapters import GPTModel
# If the `previous_chapters.py` file is not available locally,
# you can import it from the `llms-from-scratch` PyPI package.
# For details, see: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# E.g.,
# from llms_from_scratch.ch04 import GPTModel



BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-xl (1558M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

- Now, let's see the GPU memory functions in action:


---

- 现在，让我们看看 GPU 内存功能的实际运作：


In [4]:
start_memory_tracking()


model = GPTModel(BASE_CONFIG)
device = torch.device("cuda")
model.to(device)

print_memory_usage()

/home/rasbt/jupyterlab/reasoning/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


Maximum GPU memory allocated: 6.4 GB


- Additionally, let's make sure that the model runs okay by passing in some example tensor


---

- 此外，我们通过传入一些示例张量来确保模型运行正常


In [5]:
# Test if the model works (no need to track memory here)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

- Next, imagine we were pretraining the model and saving it for later use
- We skip the actual pretraining here for simplicity and just save the initialized model (but the same concept applies)


---

- 接下来，假设我们正在对模型进行预训练并将其保存以供后续使用
- 为简化流程，此处我们跳过实际的预训练步骤，仅保存初始化模型（但概念是相同的）


In [6]:
# Training code would go here...

model.train()
torch.save(model.state_dict(), "model.pth")

- Lastly, we delete the model and example tensor in the Python session to reset the GPU memory


---

- 最后，我们在Python会话中删除模型和示例张量以重置GPU内存


In [7]:
del model, test_input
cleanup()

Maximum GPU memory allocated: 0.0 GB


&nbsp;
## 3. Basic weight loading


---

&nbsp;
## 3. 基础权重加载


- Now begins the interesting part where we load the pretrained model weights
- Let's see how much GPU memory is required to load the previously saved model


---

- 现在开始加载预训练模型权重的有趣部分
- 让我们看看加载之前保存的模型需要多少GPU显存


In [8]:
# Then load pretrained weights

start_memory_tracking()

model = GPTModel(BASE_CONFIG)
model.to(device)

model.load_state_dict(
    torch.load("model.pth", map_location=device, weights_only=True)
)
model.to(device)
model.eval();

print_memory_usage()

Maximum GPU memory allocated: 12.8 GB


- Notice that the memory is 2x as large as in the previous session
- This is because we have the same model in memory twice, for a short period of time:
  - The first time via `model.to(device)`
  - The second time via the code line `model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))`; eventually, the loaded model weights will be copied into the model, and the `state_dict` will be discarded, but for a brief amount of time, we have both the main model and the loaded `state_dict` in memory
- The remaining sections focus on addressing this
- But first, let's test the model and reset the GPU memory


---

- 请注意，内存占用是上一次会话的两倍
- 这是因为在短时间内，内存中同时存在两个相同的模型：
  - 第一次通过 `model.to(device)` 加载
  - 第二次通过代码行 `model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))` 加载；最终，加载的模型权重会被复制到模型中，`state_dict` 将被丢弃，但在短暂的时间内，主模型和加载的 `state_dict` 同时存在于内存中
- 后续章节将重点讨论如何解决这个问题
- 但首先，让我们测试模型并重置 GPU 内存


In [9]:
# Test if the model works (no need to track memory here)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input
cleanup()

Maximum GPU memory allocated: 0.0 GB


- Let's test another common pattern that is very popular in practice:


---

- 让我们测试另一个实践中非常流行的常见模式：


In [10]:
start_memory_tracking()

model = GPTModel(BASE_CONFIG)

model.load_state_dict(
    torch.load("model.pth", map_location="cpu", weights_only=True)
)
model.to(device)
model.eval();

print_memory_usage()

Maximum GPU memory allocated: 6.4 GB


In [11]:
# Test if the model works (no need to track memory here)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input
cleanup()

Maximum GPU memory allocated: 0.0 GB


- So, as peak memory is concerned, it doesn't make a difference whether we instantiate the model on the device first and then use `map_location="device"` or load the weights into CPU memory first (`map_location="cpu"`) and then move it to the device


---

- 就峰值内存而言，先在设备上实例化模型再使用 `map_location="device"`，还是先将权重加载到CPU内存（`map_location="cpu"`）再转移到设备上，两者并无差异。


&nbsp;
## 4. Loading weights sequentially


---

&nbsp;
## 4. 按顺序加载权重


- One workaround for the problem of having the model weights in GPU memory twice, as highlighted in the previous section, is to load the model sequentially
- Below, we:
  - first load the model into GPU memory
  - then load the model weights into CPU memory
  - and finally copy each parameter one by one into GPU memory


---

- 针对上一节提到的模型权重在GPU内存中重复存储的问题，一种解决方法是顺序加载模型
- 下面我们将：
  - 首先把模型加载到GPU内存中
  - 然后将模型权重加载到CPU内存中
  - 最后将每个参数逐一复制到GPU内存中


In [ ]:
start_memory_tracking()

model = GPTModel(BASE_CONFIG).to(device)

state_dict = torch.load("model.pth", map_location="cpu", weights_only=True)

print_memory_usage()

# Sequentially copy weights to the model's parameters
with torch.no_grad():
    for name, param in model.named_parameters():
        if name in state_dict:
            param.copy_(state_dict[name].to(device))
        else:
            print(f"Warning: {name} not found in state_dict.")

print_memory_usage()

Maximum GPU memory allocated: 6.4 GB
Maximum GPU memory allocated: 6.7 GB


- As we can see above, the memory usage is much lower than before
- Notice that the memory increases from 6.4 to 6.7 GB because initially, we only have the model in memory, and then we have the model plus 1 parameter tensor in memory (we temporarily move the parameter tensor to the GPU so we can assign it using `".to"` the model)
- Overall, this is a significant improvement
- Again, let's briefly test the model and then reset the GPU memory for the next section


---

- 如上所示，内存使用量比之前大幅降低
- 注意内存从6.4 GB增加到6.7 GB，因为最初内存中仅有模型，随后内存中同时存在模型和1个参数张量（我们暂时将参数张量移至GPU以便通过`".to"`方法将其赋值给模型）
- 总体而言，这是显著的改进
- 接下来让我们简要测试模型，然后重置GPU内存以便进行下一章节


In [ ]:
# Test if the model works (no need to track memory here)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input, state_dict, param
cleanup()

Maximum GPU memory allocated: 0.0 GB


&nbsp;
## 5. Loading the model with low CPU memory


---

&nbsp;
## 5. 使用低CPU内存加载模型


- In the previous session, we reduced GPU memory use by loading the weights (`state_dict`) into CPU memory first before copying them one-by-one into the model
- However, what do we do if we have limited CPU memory?
- This section uses PyTorch's so-called `"meta"` device approach to load a model on machines with large GPU memory but small CPU memory
- But first, let's define a convenience function to monitor CPU memory


---

- 在上一环节中，我们通过先将权重（`state_dict`）加载到CPU内存，再逐一复制到模型的方式，降低了GPU内存占用
- 但若CPU内存有限，我们该如何应对？
- 本节将使用PyTorch的`"meta"`设备方法，在GPU内存充足但CPU内存有限的机器上加载模型
- 首先，让我们定义一个便捷函数来监控CPU内存使用情况


In [ ]:
import os
import psutil
from threading import Thread


def memory_usage_in_gb(func, *args, **kwargs):
    process = psutil.Process(os.getpid())

    # Measure the baseline memory usage before running the function
    baseline_mem = process.memory_info().rss / 1024 ** 3  # in GB

    # Start monitoring memory in a separate thread
    mem_usage = []
    done = False

    def monitor_memory():
        while not done:
            mem_usage.append(process.memory_info().rss / 1024 ** 3)  # Convert to GB
            time.sleep(0.1)

    t = Thread(target=monitor_memory)
    t.start()

    # Run the function
    func(*args, **kwargs)

    # Stop monitoring
    done = True
    t.join()

    peak_mem_usage_gb = max(mem_usage) - baseline_mem
    return peak_mem_usage_gb


- To start with, let's track the CPU memory of the sequential weight loading approach from the previous section


---

- 首先，让我们追踪上一节中顺序权重加载方法的CPU内存使用情况


In [ ]:
def load_sequentially():
    start_memory_tracking()

    model = GPTModel(BASE_CONFIG).to(device)

    state_dict = torch.load("model.pth", map_location="cpu", weights_only=True)

    print_memory_usage()

    # Sequentially copy weights to the model's parameters
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.copy_(state_dict[name].to(device))
            else:
                print(f"Warning: {name} not found in state_dict.")

    print_memory_usage()


peak_memory_used = memory_usage_in_gb(load_sequentially)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 6.4 GB
Maximum GPU memory allocated: 6.7 GB
-> Maximum CPU memory allocated: 6.3 GB


- Now, suppose we have a machine with low CPU memory but large GPU memory
- We can trade off CPU memory and GPU memory usage by introducing PyTorch's so-called "meta" device
- PyTorch's meta device is a special device type that allows you to create tensors without allocating actual memory for their data, effectively creating "meta" tensors
- This is useful for tasks like model analysis or architecture definition, where you need tensor shapes and types without the overhead of memory allocation


---

- 现在，假设我们有一台CPU内存较小但GPU内存较大的机器
- 通过引入PyTorch所谓的"meta"设备，我们可以在CPU内存和GPU内存使用之间进行权衡
- PyTorch的meta设备是一种特殊的设备类型，允许您创建张量而不为其数据分配实际内存，从而有效地创建"meta"张量
- 这对于模型分析或架构定义等任务非常有用，因为在这些任务中您需要张量的形状和类型，而无需承担内存分配的开销


In [ ]:
def load_sequentially_with_meta():
    start_memory_tracking()

    with torch.device("meta"):
        model = GPTModel(BASE_CONFIG)

    model = model.to_empty(device=device)

    state_dict = torch.load("model.pth", map_location=device, weights_only=True)

    print_memory_usage()

    # Sequentially copy weights to the model's parameters
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.copy_(state_dict[name])
            else:
                print(f"Warning: {name} not found in state_dict.")

    print_memory_usage()

peak_memory_used = memory_usage_in_gb(load_sequentially_with_meta)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 12.8 GB
Maximum GPU memory allocated: 12.8 GB
-> Maximum CPU memory allocated: 1.3 GB


- As we can see above, by creating the model on the meta-device and loading the weights directly into GPU memory, we effectively reduced the CPU memory requirements
- One might ask: "Is the sequential weight loading still necessary then, and how does that compare to the original approach?"
- Let's check the simple PyTorch weight loading approach for comparison (from the first weight loading section in this notebook):


---

- 如上所示，通过在元设备上创建模型并将权重直接加载到GPU内存中，我们有效降低了CPU内存需求
- 有人可能会问："那么顺序加载权重是否仍然必要？这与原始方法相比如何？"
- 让我们查看简单的PyTorch权重加载方法进行对比（来自本笔记本第一个权重加载部分）：


In [ ]:
def baseline():
    start_memory_tracking()

    model = GPTModel(BASE_CONFIG)
    model.to(device)

    model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
    model.to(device)
    model.eval();

    print_memory_usage()

peak_memory_used = memory_usage_in_gb(baseline)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 12.8 GB
-> Maximum CPU memory allocated: 4.4 GB


- As we can see above, the "simple" weight loading without the meta device uses more memory
- In other words, if you have a machine with limited CPU memory, you can use the meta device approach to directly load the model weights into GPU memory to reduce peak CPU memory usage


---

- 如上所示，不使用meta设备的"简单"权重加载会占用更多内存
- 换句话说，如果您的机器CPU内存有限，可以使用meta设备方法将模型权重直接加载到GPU内存中，以降低峰值CPU内存使用量


&nbsp;
## 6. Using `mmap=True` (recommmended)


---

&nbsp;
## 6. 使用 `mmap=True`（推荐）


- As an intermediate or advanced `torch.load` user, you may wonder how these approaches compare to the `mmap=True` setting in PyTorch
- The `mmap=True` setting in PyTorch enables memory-mapped file I/O, which allows the tensor to access data directly from disk storage, thus reducing memory usage by not loading the entire file into RAM if RAM is limited
- Also, see the helpful comment by [mikaylagawarecki](https://github.com/rasbt/LLMs-from-scratch/issues/402)
- At first glance, it may look less efficient than the sequential approaches above:


---

- 作为`torch.load`的中级或高级用户，您可能会好奇这些方法与PyTorch中的`mmap=True`设置相比如何
- PyTorch中的`mmap=True`设置启用了内存映射文件I/O，允许张量直接从磁盘存储访问数据，从而在内存有限时通过不将整个文件加载到RAM中来减少内存使用
- 另外，请参阅[mikaylagawarecki](https://github.com/rasbt/LLMs-from-scratch/issues/402)的有用评论
- 乍一看，这可能看起来比上述顺序方法效率低：


In [ ]:
def best_practices():
  with torch.device("meta"):
      model = GPTModel(BASE_CONFIG)

  model.load_state_dict(
      torch.load("model.pth", map_location=device, weights_only=True, mmap=True),
      assign=True
  )

  print_memory_usage()

peak_memory_used = memory_usage_in_gb(best_practices)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 6.4 GB
-> Maximum CPU memory allocated: 5.9 GB


- The reason why the CPU RAM usage is so high is that there's enough CPU RAM available on this machine
- However, if you were to run this on a machine with limited CPU RAM, the `mmap` approach would use less memory


---

- CPU RAM 使用率如此之高的原因是这台机器上有足够的 CPU RAM 可用
- 然而，如果在 CPU RAM 有限的机器上运行此程序，`mmap` 方法将使用更少的内存


&nbsp;
## 7. Other methods


---

&nbsp;
## 7. 其他方法


- This notebook is focused on simple, built-in methods for loading weights in PyTorch
- The recommended approach for limited CPU memory cases is the `mmap=True` approach explained enough
- Alternatively, one other option is a brute-force approach that saves and loads each weight tensor separately:


---

- 本笔记本专注于在PyTorch中加载权重的简单内置方法
- 对于CPU内存有限的情况，推荐使用已充分说明的 `mmap=True` 方法
- 另一种选择是采用暴力方法，分别保存和加载每个权重张量：


In [ ]:
model = GPTModel(BASE_CONFIG)
# Assume `model` is your trained model
state_dict = model.state_dict()

# Create a directory to store individual parameter files
os.makedirs("model_parameters", exist_ok=True)

# Save each parameter tensor separately
for name, param in state_dict.items():
    torch.save(param.cpu(), f"model_parameters/{name}.pt")

del model

In [ ]:
def load_individual_weights():

    start_memory_tracking()

    with torch.device("meta"):
        model = GPTModel(BASE_CONFIG)

    model = model.to_empty(device=device)

    print_memory_usage()
    param_dir = "model_parameters"

    with torch.no_grad():
        for name, param in model.named_parameters():
            weight_path = os.path.join(param_dir, f"{name}.pt")
            if os.path.exists(weight_path):
                param_data = torch.load(weight_path, map_location="cpu", weights_only=True)
                param.copy_(param_data)
                del param_data  # Free memory
            else:
                print(f"Warning: {name} not found in {param_dir}.")

    print_memory_usage()


peak_memory_used = memory_usage_in_gb(load_individual_weights)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 6.4 GB
Maximum GPU memory allocated: 6.4 GB
-> Maximum CPU memory allocated: 0.3 GB
